# HW Review - Regression Unit

This notebook addresses common conceptual gaps from HW1 and HW2. It is not a solution key - it covers the *ideas* that tripped people up, with worked examples using different data than the assignments. Use it as a self-study resource before the exam.

## HW1: Regression Fundamentals

### Visualizing bias and variance

Before diving into specific issues, it helps to have a concrete picture of what bias and variance *look like*. Imagine training the same model on many different random samples from the same population. Each sample produces a slightly different prediction for the same input - those predictions form a cloud.

- *Bias* is how far the center of that cloud is from the true value (systematic error).
- *Variance* is how spread out the cloud is (sensitivity to the training sample).

The dartboard figure below shows all four combinations. Each blue dot is the prediction from one training sample; the red bullseye is the true value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

configs = [
    ('Low Bias\nLow Variance',   0.0, 0.0, 0.15),
    ('Low Bias\nHigh Variance',  0.0, 0.0, 0.6),
    ('High Bias\nLow Variance',  0.7, 0.5, 0.15),
    ('High Bias\nHigh Variance', 0.7, 0.5, 0.6),
]

for ax, (title, bx, by, spread) in zip(axes, configs):
    # Draw target rings
    for r, alpha in [(1.5, 0.08), (1.1, 0.12), (0.7, 0.18), (0.3, 0.3)]:
        circle = plt.Circle((0, 0), r, fill=True, color='firebrick', alpha=alpha)
        ax.add_patch(circle)
        circle_edge = plt.Circle((0, 0), r, fill=False, color='darkred', linewidth=0.5, alpha=0.4)
        ax.add_patch(circle_edge)

    # Bullseye center
    ax.plot(0, 0, 'o', color='firebrick', markersize=5, zorder=3)

    # Predictions (dots scattered around bias offset)
    n_dots = 15
    px = bx + np.random.randn(n_dots) * spread
    py = by + np.random.randn(n_dots) * spread
    ax.plot(px, py, 'o', color='steelblue', markersize=6, alpha=0.8, zorder=4)

    # Mean of predictions
    ax.plot(px.mean(), py.mean(), '+', color='navy', markersize=12,
            markeredgewidth=2, zorder=5)

    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Bias-Variance Trade-off', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

If this looks familiar, it should - it is the same decomposition as *accuracy vs. precision* in measurement science. Accuracy is how close the mean measurement is to the true value (bias); precision is how tightly repeated measurements cluster (variance). The same 2x2 grid appears in every introductory lab course. In ML, we are doing the same thing with model predictions instead of instrument readings.

The ideal is the top-left: low bias *and* low variance. The trade-off arises because increasing model complexity tends to move you right (more variance) while decreasing bias, and regularization moves you left (less variance) at the cost of some bias. The goal is to find the sweet spot - not to eliminate one at the expense of the other.

### Bias is already zero when the model is correctly specified

Several students wrote that bias "decreases" as model flexibility increases when the true relationship is linear. This confuses two different scenarios.

When the truth is linear and you fit a linear model, the model is already correctly specified. Bias is *zero* - not "low," not "decreasing," but zero. There is no systematic error because the model can represent the true function exactly.

Bias decreases with flexibility only when the model is *misspecified* - for example, fitting a linear model to a quadratic truth. In that case, adding polynomial terms reduces bias by letting the model capture curvature it was previously missing.

The key question is: **does the model class contain the true function?** If yes, bias is zero regardless of flexibility. If no, increasing flexibility can reduce bias - but at the cost of variance.

The two panels below contrast these scenarios. On the left, a linear model fits a linear truth - the fit is exact, so bias is zero everywhere. On the right, a linear model fits a quadratic truth - the systematic gap (shaded) is bias.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 50
x = np.linspace(0, 1, n)

# True linear relationship
y_true = 2 + 3 * x

# Case 1: Linear model fits linear truth → bias = 0
# Case 2: Linear model fits quadratic truth → bias > 0
y_true_quad = 2 + 3 * x - 4 * x**2

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: correctly specified
ax = axes[0]
ax.plot(x, y_true, 'k-', linewidth=2, label='Truth (linear)')
ax.plot(x, y_true, 'r--', linewidth=2, label='Linear fit (E[ŷ])')
ax.set_title('Correctly Specified\nBias = 0 everywhere')
ax.legend(fontsize=9)
ax.set_xlabel('x')

# Right: misspecified
from numpy.polynomial import polynomial as P
coeffs = np.polyfit(x, y_true_quad, 1)
y_linear_fit = np.polyval(coeffs, x)

ax = axes[1]
ax.plot(x, y_true_quad, 'k-', linewidth=2, label='Truth (quadratic)')
ax.plot(x, y_linear_fit, 'r--', linewidth=2, label='Linear fit (E[ŷ])')
ax.fill_between(x, y_true_quad, y_linear_fit, alpha=0.2, color='red', label='Bias')
ax.set_title('Misspecified\nBias > 0 (shaded gap)')
ax.legend(fontsize=9)
ax.set_xlabel('x')

plt.tight_layout()
plt.show()

The left panel shows zero bias - the linear model *is* the truth. The right panel shows nonzero bias - the linear model systematically misses the curvature. Adding a quadratic term would eliminate this bias, but adding flexibility to the left scenario would only increase variance with no bias benefit.

### Predictor scale matters for interpreting coefficients

A coefficient of 0.01 looks tiny. But if the predictor ranges from 0 to 1,000, then a one-unit change in the predictor produces a 0.01-unit change in the response - while multiplying that coefficient by a realistic change in the predictor (say, 500 points) gives a 5-unit change. Whether a coefficient is "practically significant" depends on the scale of its predictor, not just the magnitude of the coefficient.

The general principle: **multiply the coefficient by a realistic change in the predictor** to assess practical significance. The example below shows two predictors with very different coefficient magnitudes that produce similar practical impacts once you account for scale.

In [ ]:
# Example: predicting salary from years of experience and test score
# Both coefficients are "small" but operate on very different scales

coef_experience = 1.5   # salary increase per year of experience
coef_test_score = 0.02  # salary increase per test-score point

# Realistic ranges
experience_range = 10    # someone might have 0-10 years
test_score_range = 500   # scores might range 200-700

print("Practical impact over realistic predictor ranges:")
print(f"  Experience: {coef_experience} × {experience_range} years = {coef_experience * experience_range:.1f} salary units")
print(f"  Test Score: {coef_test_score} × {test_score_range} points = {coef_test_score * test_score_range:.1f} salary units")
print()
print("The 'tiny' test score coefficient has the same practical impact")
print("as the 'large' experience coefficient once you account for scale.")

Build the habit of translating every coefficient into response units. Instead of "ShelveLoc_Good has a positive coefficient," say "products on good shelves sell approximately 4,850 more units than products on bad shelves, holding all else equal." The coefficient alone is just a number - multiplying it by the predictor's units and stating the result in the response's units is what makes it an interpretation.

### Interpreting interaction terms

Interaction terms change the slope of one predictor depending on the value of another. The sign of the interaction coefficient tells you the *direction* of this modification.

Consider a model predicting sales from Price and ShelfLocation (Good vs. Bad):

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{Price} + \beta_2 \cdot \text{Good} + \beta_3 \cdot \text{Price} \times \text{Good}$$

If $\beta_1 = -0.05$ (price hurts sales) and $\beta_3 = +0.02$ (positive interaction), the interpretation is:

- Bad shelves: slope of Price = $\beta_1 = -0.05$
- Good shelves: slope of Price = $\beta_1 + \beta_3 = -0.05 + 0.02 = -0.03$

The positive interaction means good shelf placement *reduces* the negative price effect. Products on good shelves are **less** price-sensitive, not more. A common mistake is reading the positive sign as "price hurts more for good shelves" - it's the opposite.

The plot below makes this concrete: both lines slope downward (higher price = lower sales), but the good-shelf line is shallower - products on good shelves lose fewer sales per dollar of price increase.

In [ ]:
price = np.linspace(50, 150, 100)

# Coefficients
b0, b1, b2, b3 = 10, -0.05, 3, 0.02

sales_bad = b0 + b1 * price
sales_good = (b0 + b2) + (b1 + b3) * price

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(price, sales_bad, 'b-', linewidth=2, label=f'Bad shelf (slope = {b1})')
ax.plot(price, sales_good, 'g-', linewidth=2, label=f'Good shelf (slope = {b1 + b3})')
ax.set_xlabel('Price')
ax.set_ylabel('Sales')
ax.set_title('Interaction Effect: Good Shelves Reduce Price Sensitivity')
ax.legend()
ax.annotate('Steeper slope = more\nprice-sensitive', xy=(130, sales_bad[-30]),
            fontsize=9, color='blue', ha='center')
ax.annotate('Shallower slope = less\nprice-sensitive', xy=(130, sales_good[-30]),
            fontsize=9, color='green', ha='center')
plt.tight_layout()
plt.show()

To interpret any interaction: write out the separate equations for each group, then compare the slopes. The interaction coefficient is the *difference* between the slopes.

### Writing strong interpretations

The single biggest differentiator between strong and average submissions is interpretive depth. Here is the same result interpreted two ways:

*Weak:* "Ridge has a higher test R² than OLS, so regularization helps."

*Strong:* "Ridge improved test R² from 0.41 to 0.46 (a 12% relative gain) while train R² dropped slightly from 0.57 to 0.52. This is consistent with OLS overfitting due to multicollinearity among the log-transformed application variables - OLS assigns large, opposing coefficients to correlated features (e.g., log_Apps at +39 and log_Accept at -42), while Ridge shrinks both toward zero, reducing variance at a small bias cost."

The strong version does three things the weak version doesn't:

1. *Cites specific numbers* from the actual output
2. *Names the mechanism* (multicollinearity inflating coefficient variance)
3. *Connects to specific features* in the data (log_Apps, log_Accept)

This applies to every interpretation question. When you see a pattern in your results, ask: *what specific numbers show this pattern, and what mechanism explains it?*

This is also what good research papers and scientific writing look like - claims backed by specific evidence and mechanistic explanations. It is what I expect in a graduate course, and developing this habit will serve you well beyond this class.

## HW2: Model Evaluation and Selection

### Why cross-validate at all?

A model that is trained on a dataset can always be evaluated on that same dataset, and the result will look good - often misleadingly good. This is because the model has already seen every example during training. If it has enough flexibility, it can memorize patterns in the training data that are just noise - random quirks of that particular sample that won't appear in new data. Training R² (or training accuracy, training MSE, etc.) reflects how well the model fits the data it has already seen, which is not the same as how well it will perform on data it hasn't.

Cross-validation solves this by repeatedly holding out a portion of the data that the model never sees during training, then measuring performance on that held-out portion. This gives you an honest estimate of how the model will perform on genuinely new data. The gap between training performance and cross-validated performance is itself informative - a large gap signals overfitting.

This is why questions like "which model generalizes best?" cannot be answered by comparing training R² values. A model with training R² of 0.95 might cross-validate at 0.60, while a simpler model with training R² of 0.80 might cross-validate at 0.75. The simpler model is better despite looking worse on the training data. Cross-validation is how you see past the illusion of training performance.

### Data regime framing: is your dataset "big"?

The UEOD paper argues that simple models with massive data can outperform complex models with less data. The key word is *massive* - millions or billions of examples, not hundreds.

When interpreting results on a dataset like College (n=777, p=50), you are in the **data-scarce regime** where UEOD does *not* apply. This is precisely the setting where:

- Regularization matters (you don't have enough data to let OLS estimate 50 coefficients freely)
- Cross-validation is essential (small n means high variance in performance estimates)
- Simpler models are preferred for parsimony, not because data overwhelms complexity

The question to ask before interpreting any result: **how much data do I have relative to the number of parameters?** With n/p ≈ 15, you are firmly in the regime where variance dominates and regularization is a substitute for the data you don't have.

### What StandardScaler leakage actually means

The majority of you recognized that you should `fit` the scaler on training data and `transform` test data with the same parameters. But *what information leaks* when you fit on everything?

To make this concrete, the code below simulates training and test sets with different distributions and compares scaler parameters when fit on all data vs. training data only.

In [ ]:
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# Simulate: training data has mean ~50, test data has mean ~70
# (imagine a dataset where test examples come from a slightly different distribution)
X_train = np.random.normal(50, 10, size=(100, 1))
X_test = np.random.normal(70, 10, size=(20, 1))

# WRONG: fit on everything
X_all = np.vstack([X_train, X_test])
scaler_wrong = StandardScaler().fit(X_all)

# RIGHT: fit on train only
scaler_right = StandardScaler().fit(X_train)

print("Scaler parameters:")
print(f"  Fit on ALL data:   mean={scaler_wrong.mean_[0]:.1f}, std={scaler_wrong.scale_[0]:.1f}")
print(f"  Fit on TRAIN only: mean={scaler_right.mean_[0]:.1f}, std={scaler_right.scale_[0]:.1f}")
print()
print("Impact on training data (first 3 samples):")
print(f"  Raw values:        {X_train[:3, 0].round(1)}")
print(f"  Scaled (wrong):    {scaler_wrong.transform(X_train[:3]).round(3).flatten()}")
print(f"  Scaled (right):    {scaler_right.transform(X_train[:3]).round(3).flatten()}")
print()
print("The test set's higher mean pulled the scaler's mean UP by",
      f"{scaler_wrong.mean_[0] - scaler_right.mean_[0]:.1f}.")
print("This shifts ALL training z-scores downward - the model trains on")
print("subtly different features than it would have without the leak.")

The leak is subtle: test-set statistics (mean, variance) shift the scaler's parameters, which changes *every* z-score in the training data. The model then trains on features that were influenced by test data it shouldn't have seen. In practice, this makes performance estimates slightly optimistic - you think your model generalizes better than it actually does.

### How Pipelines prevent leakage inside CV folds

When you call `cross_val_score(pipeline, X, y, cv=5)`, here is what happens inside each fold:

```text
Fold 1:
  Train = folds 2,3,4,5 | Validation = fold 1

  pipeline.fit(X_train_fold, y_train_fold)
    → scaler.fit(X_train_fold)          # learns mean/std from folds 2-5 ONLY
    → scaler.transform(X_train_fold)    # scales training data
    → model.fit(X_train_scaled, y_train_fold)

  pipeline.predict(X_val_fold)
    → scaler.transform(X_val_fold)      # uses folds 2-5 parameters
    → model.predict(X_val_scaled)

Fold 2:
  Train = folds 1,3,4,5 | Validation = fold 2

  pipeline.fit(X_train_fold, y_train_fold)
    → scaler.fit(X_train_fold)          # DIFFERENT mean/std (folds 1,3-5)
    → ...
```

The scaler is **re-fit from scratch** on each fold's training portion. Fold 1's validation data never influences the scaler that processes it.

Compare to pre-scaling:

```text
scaler.fit(X)                    # sees ALL data including every validation fold
X_scaled = scaler.transform(X)
cross_val_score(model, X_scaled, y, cv=5)
  → Fold 1 validation data was already used to compute mean/std
  → Every fold's "test" data influenced its own preprocessing
```

This is why Pipeline + CV produces honest estimates: the validation fold is truly unseen at every step, including preprocessing.

To make this concrete, here is a manual implementation of what `cross_val_score` does internally when you pass it a Pipeline. Notice that the scaler is re-fit on each fold's training portion - the `scaler.mean_[0]` value changes every fold, confirming that no fold shares preprocessing parameters with another.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Synthetic data for demonstration
np.random.seed(42)
X_demo = np.random.randn(200, 5)
y_demo = X_demo @ [3, -2, 1.5, -1, 0.5] + np.random.randn(200) * 2

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Manual CV with proper scaler handling (what Pipeline does):\n")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_demo)):
    X_train_fold, X_val_fold = X_demo[train_idx], X_demo[val_idx]
    y_train_fold, y_val_fold = y_demo[train_idx], y_demo[val_idx]

    # This is what Pipeline does internally:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold)  # fit on THIS fold's train
    X_val_scaled = scaler.transform(X_val_fold)           # transform only

    model = Ridge(alpha=1.0)
    model.fit(X_train_scaled, y_train_fold)
    score = model.score(X_val_scaled, y_val_fold)
    print(f"  Fold {fold+1}: R² = {score:.3f}, scaler mean[0] = {scaler.mean_[0]:.3f}")

Each fold's scaler sees different data, so it learns different parameters. That is exactly the point - and exactly what you lose if you pre-scale before calling `cross_val_score`.

### Regularization paths: shrinking vs. zeroing

Ridge and Lasso both penalize large coefficients, but they do it differently. Watching how coefficients evolve across the penalty strength (alpha) makes the distinction visual.

The code below generates synthetic data with 5 informative features, 5 correlated copies, and 5 pure noise features, then sweeps alpha from 0.01 to 10,000 and plots how each coefficient responds.

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.datasets import make_regression

# Generate data with some correlated and some irrelevant features
np.random.seed(42)
n, p = 200, 15
X = np.random.randn(n, p)
# Make features 0-4 informative, 5-9 correlated with 0-4, 10-14 noise
X[:, 5:10] = X[:, 0:5] + np.random.randn(n, 5) * 0.3
true_coefs = np.zeros(p)
true_coefs[:5] = [3, -2, 1.5, -1, 0.5]
y = X @ true_coefs + np.random.randn(n) * 2

alphas = np.logspace(-2, 4, 100)
feature_names = [f'x{i}' for i in range(p)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Ridge paths
ridge_coefs = np.array([Ridge(alpha=a).fit(X, y).coef_ for a in alphas])
ax = axes[0]
for j in range(p):
    style = '-' if j < 5 else ('--' if j < 10 else ':')
    alpha_val = 0.6 if j < 10 else 0.3
    ax.plot(np.log10(alphas), ridge_coefs[:, j], style, alpha=alpha_val)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.set_xlabel('log₁₀(α)')
ax.set_ylabel('Coefficient value')
ax.set_title('Ridge: All coefficients shrink toward zero\nbut none reach exactly zero')

# Lasso paths
lasso_coefs = np.array([Lasso(alpha=a, max_iter=10000).fit(X, y).coef_ for a in alphas])
ax = axes[1]
for j in range(p):
    style = '-' if j < 5 else ('--' if j < 10 else ':')
    alpha_val = 0.6 if j < 10 else 0.3
    ax.plot(np.log10(alphas), lasso_coefs[:, j], style, alpha=alpha_val)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.set_xlabel('log₁₀(α)')
ax.set_ylabel('Coefficient value')
ax.set_title('Lasso: Coefficients hit zero at different α\n(automatic feature selection)')

plt.tight_layout()
plt.show()

# Count nonzero at a few alpha values
print("Number of nonzero Lasso coefficients at different alpha values:")
for a in [0.01, 0.1, 1.0, 10.0]:
    coefs = Lasso(alpha=a, max_iter=10000).fit(X, y).coef_
    print(f"  α = {a:>5}: {np.sum(np.abs(coefs) > 1e-6):>2} / {p} features retained")

This is the continuous-vs-discrete distinction that matters for understanding regularization:

- *Ridge* applies continuous shrinkage - every coefficient gets smaller as alpha increases, but none ever reach exactly zero. Ridge keeps all features but reduces their influence.
- *Lasso* performs continuous shrinkage that produces discrete outcomes - the L1 penalty's geometry forces coefficients to hit exactly zero at feature-specific alpha thresholds. This is automatic variable selection through a continuous optimization process.
- *Best subset selection* makes discrete include/exclude decisions by exhaustive search. It doesn't shrink - features are either fully in or fully out.

Lasso is preferred in practice because it combines both shrinkage (for variance reduction) and selection (for interpretability) in a single, computationally efficient optimization.

### The curse of dimensionality

"Curse of dimensionality" appeared in many answers but few explanations went beyond "distances become less meaningful." Here is a more concrete way to think about it.

Imagine you have 20 data points on a line segment of length 1. They are fairly close together - the data covers the space reasonably well. Now scatter those same 20 points across a plane of the same size (x,y in [0,1]). The space is larger, so the points spread out and the average distance between them grows. Scatter them across a unit cube and they spread out even more. Each time you add a dimension, the volume of the space grows exponentially, but you still have only 20 points trying to fill it.

This is the core problem: with a fixed number of training examples, adding features is like scattering the same handful of points into a larger and larger room. The data becomes increasingly sparse, and *every* model suffers - there simply isn't enough information to learn the structure of the space. More features means more parameters to estimate, more possible interactions to capture, and more room for the model to find spurious patterns that don't generalize. This is true for linear regression, trees, neural networks, and everything else. KNN just makes the problem most visible because it relies explicitly on distances, but the underlying issue is about data sparsity, not any particular algorithm.

The figure below makes this visible. We place the same 30 points in 1D, 2D, and 3D and watch how the gaps between them grow.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)
n = 30

fig = plt.figure(figsize=(14, 4))

# 1D: points on a line
ax1 = fig.add_subplot(131)
x1 = np.random.uniform(0, 1, n)
ax1.scatter(x1, np.zeros(n), s=40, color='steelblue', zorder=3)
ax1.set_xlim(-0.1, 1.1)
ax1.set_ylim(-0.3, 0.3)
ax1.set_yticks([])
ax1.set_title(f'1D: {n} points on a line\nPoints are close together')
ax1.set_xlabel('x₁')

# 2D: points in a square
ax2 = fig.add_subplot(132)
x2 = np.random.uniform(0, 1, (n, 2))
ax2.scatter(x2[:, 0], x2[:, 1], s=40, color='steelblue', zorder=3)
ax2.set_xlim(-0.1, 1.1)
ax2.set_ylim(-0.1, 1.1)
ax2.set_title(f'2D: same {n} points in a square\nGaps are appearing')
ax2.set_xlabel('x₁')
ax2.set_ylabel('x₂')
ax2.set_aspect('equal')

# 3D: points in a cube
ax3 = fig.add_subplot(133, projection='3d')
x3 = np.random.uniform(0, 1, (n, 3))
ax3.scatter(x3[:, 0], x3[:, 1], x3[:, 2], s=40, color='steelblue')
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.set_zlim(0, 1)
ax3.set_title(f'3D: same {n} points in a cube\nMostly empty space')
ax3.set_xlabel('x₁')
ax3.set_ylabel('x₂')
ax3.set_zlabel('x₃')

plt.tight_layout()
plt.show()

We can only visualize up to 3 dimensions, but the pattern continues: in 50 or 100 dimensions, 30 points are hopelessly sparse. Every point is far from every other point, and there are vast empty regions with no training data at all.

This has two concrete consequences that we can measure. First, the average distance between any two points grows steadily with the number of dimensions - your neighbors are no longer nearby. Second, as dimensions increase, virtually all points end up near the edges of the space rather than in the interior, so predictions are based on extrapolation rather than interpolation.

In [ ]:
np.random.seed(42)
n_points = 500
dims = [1, 2, 3, 5, 10, 25, 50, 100, 250, 500]

avg_dists = []
border_fracs = []

for d in dims:
    points = np.random.uniform(0, 1, size=(n_points, d))

    # Average pairwise distance (sample 5000 random pairs for speed)
    idx = np.random.choice(n_points, size=(5000, 2), replace=True)
    pair_dists = np.sqrt(np.sum((points[idx[:, 0]] - points[idx[:, 1]]) ** 2, axis=1))
    avg_dists.append(pair_dists.mean())

    # Fraction of points within 0.01 of any border
    near_border = np.any((points < 0.01) | (points > 0.99), axis=1)
    border_fracs.append(near_border.mean())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: average pairwise distance
ax = axes[0]
ax.plot(dims, avg_dists, 'bo-', linewidth=2, markersize=5)
ax.set_xlabel('Number of dimensions')
ax.set_ylabel('Average distance between points')
ax.set_title('Points get farther apart')
ax.set_xscale('log')

# Right: fraction of points near border
ax = axes[1]
ax.plot(dims, border_fracs, 'ro-', linewidth=2, markersize=5)
ax.set_xlabel('Number of dimensions')
ax.set_ylabel('Fraction of points near a border')
ax.set_title('Points crowd toward the edges')
ax.set_xscale('log')
ax.set_ylim(-0.05, 1.05)
ax.axhline(y=1.0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"In 1D:   avg distance between points = {avg_dists[0]:.2f}")
print(f"In 2D:   avg distance = {avg_dists[1]:.2f}")
print(f"In 500D: avg distance = {avg_dists[-1]:.1f}")
print()
print(f"In 2D:   {border_fracs[1]:.0%} of points are near a border")
print(f"In 500D: {border_fracs[-1]:.0%} of points are near a border")

This is why high-dimensional datasets are so challenging. Every model needs training points that adequately cover the input space, and in high dimensions the space is so vast that even large datasets are sparse. KNN is the most dramatic casualty because it depends on finding genuinely close neighbors, but linear models also suffer - they have more coefficients to estimate from the same amount of data, which is exactly the variance problem that regularization addresses.